# Dependencies

In [1]:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
# !pip install websockets nest_asyncio pandas

# Physics Engine & Constants

In [1]:
import re
import json
import asyncio
import nest_asyncio
import pandas as pd
import numpy as np
from websockets.asyncio.client import connect as ws_connect

nest_asyncio.apply()

HF_WS_URL = "wss://mukundnjoy-heat-treatment-scheduler.hf.space/ws"
TASK_NAME = "easy-bake"

CONST = {
    "temp_max": 600.0,
    "r_max_clip": 30.0,
    "r_target": 12.5,
    "r_target_min": 10.0,
    "r_target_max": 15.0,
    "growth_threshold": 176.0,
    "temp_melt": 502.0,
}

async def _run_episode_ws(text: str) -> dict:
    matches = re.findall(r"^([0-5])\s*,\s*(\d+(?:\.\d+)?)$", text, re.MULTILINE)
    if not matches:
        return {"reward": -500.0, "error": "no_valid_actions", "radius_nm": 0.0, "core_temp_C": 20.0, "max_temp_C": 20.0}

    try:
        async with ws_connect(HF_WS_URL, open_timeout=15, close_timeout=5) as ws:
            await ws.send(json.dumps({"type": "reset", "data": {"task_name": TASK_NAME}}))
            reset_resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))

            if reset_resp.get("type") == "error":
                return {"reward": -500.0, "error": f"reset_error"}

            total_reward = 0.0
            steps_taken = 0
            final_temp = 0.0
            final_r = 0.0
            max_temp_reached = 0.0
            prev_temp = 20.0
            prev_r = 0.0
            entered_growth_phase = False

            for match in matches:
                action_num = int(match[0])
                safe_duration = min(max(float(match[1]), 1.0), 600.0)

                if action_num == 5:
                    break

                await ws.send(json.dumps({
                    "type": "step",
                    "data": {"action_num": action_num, "duration_minutes": safe_duration}
                }))
                step_resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=15))

                if step_resp.get("type") == "error":
                    total_reward -= 100.0
                    break

                data = step_resp.get("data", {})
                total_reward += data.get("reward", 0.0) or 0.0
                steps_taken += 1

                obs = data.get("observation", {})
                final_temp = obs.get("temperature", 0.0) * CONST["temp_max"]
                final_r = obs.get("radius", 0.0) * CONST["r_max_clip"]

                if final_temp > max_temp_reached:
                    max_temp_reached = final_temp

                temp_delta = max(final_temp - prev_temp, 0.0)
                if final_temp < CONST["growth_threshold"]:
                    total_reward += temp_delta * 1.0
                elif final_temp < CONST["temp_melt"] - 100:
                    total_reward += 5.0

                r_delta = max(final_r - prev_r, 0.0)
                if r_delta > 0:
                    total_reward += r_delta * 20.0

                prev_temp = final_temp
                prev_r = final_r

                if final_temp >= CONST["growth_threshold"]:
                    entered_growth_phase = True

                if data.get("done", False):
                    break

            if entered_growth_phase:
                total_reward += 150.0
            else:
                total_reward -= 150.0

            if steps_taken < 5:
                total_reward -= (5 - steps_taken) * 30.0

            if final_temp < CONST["temp_melt"]:
                if final_r > CONST["r_target_max"]:
                    total_reward -= 150.0
                else:
                    progress = min(final_r / CONST["r_target"], 1.0)
                    total_reward += progress * 200.0
                    if final_r >= CONST["r_target_min"]:
                        total_reward += 50.0
                    if CONST["r_target_min"] <= final_r <= CONST["r_target_max"]:
                        total_reward += 100.0

            import math
            if math.isnan(total_reward) or math.isinf(total_reward):
                total_reward = -500.0
            else:
                total_reward = max(-500.0, min(500.0, total_reward))

            return {
                "reward": total_reward,
                "radius_nm": final_r,
                "core_temp_C": final_temp,
                "max_temp_C": max_temp_reached,
                "recipe_steps": steps_taken,
                "error": None
            }

    except Exception as e:
        return {"reward": -500.0, "error": str(e), "radius_nm": 0.0, "core_temp_C": 20.0, "max_temp_C": 20.0}

# Inference and Evaluation Loop

In [2]:
from unsloth import FastLanguageModel

# Load Base Model (No LoRA adapter)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

system_prompt = """You are an expert Metallurgical Process Engineer controlling a precipitation hardening furnace.
Your objective: Grow nanoprecipitates to the exact target radius (12.5 nm) without melting the material.
CRITICAL PHYSICS CONSTRAINTS:
1. The material starts at 20°C. Precipitates WILL NOT GROW until the core temperature reaches ~176°C.
2. Your maximum heating action (4) increases the furnace by +50°C per step.
3. WARNING: The material melts at 502°C! Stay well below this temperature.
STRATEGY:
Heat aggressively with action 4 and short durations (2 min) until the furnace is around 300-350°C (safely in the growth phase). Then hold with action 2 and long durations (60-120 min) to let precipitates grow. Monitor the radius — when it approaches 12.5 nm, cool down with action 1 or 0.
Format each step on a new line as exactly TWO numbers separated by a comma: [Action_Num, Duration_Minutes]
ACTION_NUM (0-5):
    0: Aggressive cooling (-50°C)
    1: Gentle cooling (-10°C)
    2: Hold furnace temperature (0°C change)
    3: Gentle heating (+10°C)
    4: Aggressive heating (+50°C)
    5: TERMINATE EPISODE (use when radius is at target)
DURATION_MINUTES: (1.0 to 600.0)
EXAMPLE RECIPE:
4, 2.0
4, 2.0
4, 2.0
4, 2.0
4, 2.0
4, 2.0
2, 120.0
1, 30.0
5, 1.0
Respond with ONLY your sequence of steps."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Context: Al-2024 in Lab Scale. Target: 12.500 nm. Generate your complete thermal recipe."},
    {"role": "assistant", "content": ""}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

# Evaluation Loop
num_evals = 20
results = []
loop = asyncio.get_event_loop()

print(f"Starting {num_evals} offline evaluations on Base Model...")

for i in range(num_evals):
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, do_sample=True)
    generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    # Run through physics engine
    metrics = loop.run_until_complete(_run_episode_ws(generated_text))
    results.append(metrics)
    print(f"Run {i+1}: Reward: {metrics['reward']:.1f} | Radius: {metrics['radius_nm']:.2f} nm | Max Temp: {metrics['max_temp_C']:.1f}°C")

# --- Analysis & Output ---
df = pd.DataFrame(results)

success_mask = (df["radius_nm"] >= CONST["r_target_min"]) & (df["radius_nm"] <= CONST["r_target_max"]) & (df["core_temp_C"] < CONST["temp_melt"])
melt_mask = df["max_temp_C"] >= CONST["temp_melt"]

success_rate = success_mask.mean() * 100
melt_rate = melt_mask.mean() * 100
avg_reward = df["reward"].mean()
mean_radius = df["radius_nm"].mean()

print("\n" + "="*40)
print("🎯 BASELINE MODEL EVALUATION RESULTS 🎯")
print("="*40)
print(f"Total Runs: {num_evals}")
print(f"Success Rate (10-15 nm):  {success_rate:.1f}%")
print(f"Melt Rate (> 502°C):      {melt_rate:.1f}%")
print(f"Average Reward:           {avg_reward:.1f}")
print(f"Mean Final Radius:        {mean_radius:.2f} nm")
print("="*40)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting 20 offline evaluations on Base Model...


/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/user/miniconda/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Run 17: Reward: -500.0 | Radius: 0.00 nm | Max Temp: 20.0°C


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 18: Reward: -291.0 | Radius: 0.00 nm | Max Temp: 91.1°C


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 19: Reward: -149.9 | Radius: 0.00 nm | Max Temp: 82.3°C
Run 20: Reward: -181.7 | Radius: 0.00 nm | Max Temp: 29.6°C

🎯 BASELINE MODEL EVALUATION RESULTS 🎯
Total Runs: 20
Success Rate (10-15 nm):  10.0%
Melt Rate (> 502°C):      0.0%
Average Reward:           -153.7
Mean Final Radius:        4.47 nm
